# 02 — Regression: complexity → LOS, judge effects

Tests the main research question using OLS on closed civil cases.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path("..").resolve()
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(ROOT / "data" / "case_features.parquet")
df = df[df["los_days"].notna()].copy()
df["log_los"] = np.log1p(df["los_days"])
print(f"cases: {len(df):,}")

In [ ]:
# Top nature_of_suit categories to avoid huge dummy matrix
top_suit = df["nature_suit"].value_counts().head(15).index
df["nature_suit_top"] = df["nature_suit"].where(df["nature_suit"].isin(top_suit), "Other")

# Judges with enough cases
judge_counts = df["District_Judge"].value_counts()
top_judges = judge_counts[judge_counts >= 200].index
df_j = df[df["District_Judge"].isin(top_judges)].copy()
print(f"cases after judge filter (>=200 cases/judge): {len(df_j):,}")

In [ ]:
train, test = train_test_split(df_j, test_size=0.2, random_state=42)

f1 = "log_los ~ complexity_index + C(nature_suit_top)"
f2 = "log_los ~ complexity_index + C(nature_suit_top) + C(District_Judge)"
f3 = "log_los ~ complexity_index * C(District_Judge) + C(nature_suit_top)"

m1 = smf.ols(f1, data=train).fit()
m2 = smf.ols(f2, data=train).fit()
m3 = smf.ols(f3, data=train).fit()

def eval_model(model, data, name):
    pred_log = model.predict(data)
    pred = np.expm1(pred_log)
    y = data["los_days"]
    return {
        "model": name,
        "R2": r2_score(y, pred),
        "MAE": mean_absolute_error(y, pred),
        "RMSE": mean_squared_error(y, pred) ** 0.5,
    }

results = pd.DataFrame([
    eval_model(m1, test, "M1: complexity + suit"),
    eval_model(m2, test, "M2: + judge FE"),
    eval_model(m3, test, "M3: complexity × judge"),
])
display(results)

In [ ]:
print(m1.summary().tables[1])
print("\nComplexity coefficient (M1):", m1.params.get("complexity_index"))

In [ ]:
# Residuals vs complexity by judge (sample)
test = test.copy()
test["pred_los"] = np.expm1(m2.predict(test))
test["residual"] = test["los_days"] - test["pred_los"]

plot_df = test.sample(min(3000, len(test)), random_state=42)
fig, ax = plt.subplots(figsize=(8, 4))
for judge, g in plot_df.groupby("District_Judge"):
    if len(g) < 30:
        continue
    ax.scatter(g["complexity_index"], g["residual"], alpha=0.15, s=8, label=None)
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("complexity_index")
ax.set_ylabel("residual LOS (days)")
ax.set_title("M2 residuals vs complexity (sample)")
fig.tight_layout()
fig.savefig(FIG / "05_regression_residuals.png", dpi=120)
plt.show()

## Document in `docs/PROJECT_LOG.md` Step 4

- Does higher complexity predict longer LOS? (sign of `complexity_index` in M1)
- Does adding judges improve fit? (M2 vs M1 R²)
- Is the complexity slope different by judge? (M3 vs M2)